<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/HW1_stairwell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **🛠 INSTRUCTOR DRAFT, AI-AUTHORED, REMOVE THIS BANNER BEFORE DISTRIBUTING**
>
> *AI-drafted by Claude (Anthropic) on 2026-05-08 for review by the instructor. All section text, prompts, and code patterns require an instructor pass. Once the instructor edits and signs off, this banner can be deleted and the notebook is owned by the instructor.*

> **Do this first: File > Save a copy in Drive.**
> You are viewing a shared notebook. Anything you type here is NOT saved.
> Save your own copy now, work only in the copy, and rename it
> `HW1_LASTNAME.ipynb` (exact filename and due date in the assignment
> calendar on Brightspace).

# HW1: Read the Stairwell

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Instructor:** Dax Soule (dax.soule@qc.cuny.edu)
**Due:** Wednesday, September 23, 2026, 11:59 PM
**Submit:** This notebook (`.ipynb`) via Brightspace

**Estimated time: about 2 hours across 6 parts.**

## What you will do

1. Use Python to compute Earth's gravity from the anchor equation we wrote on the board: `g = GM / r²`.
2. Load a real gravimeter dataset collected in a stairwell.
3. Make two plots: gravity over time (the *drift*) and gravity over elevation (the *free-air trend*).
4. Fit the instrument drift through the repeated ground-level reads, subtract it from every reading, then fit the *drift-corrected* gravity against elevation to measure the free-air gradient. Report it with an uncertainty.
5. Derive the free-air gradient yourself from `g = GM/r²` using the binomial expansion on the Taylor & Binomial card (in the Math Reference Cards PDF on the Resources page).
6. Reflect on which assumptions in `g = GM/r²` this dataset just exposed.

## What you will hand in

This same notebook, with your code, your plots, and your short answers filled in. Save it as `HW1_LASTNAME.ipynb` and upload to Brightspace.

## Honor pledge (CUNY CoPilot policy)

Per the course AI policy, **CUNY CoPilot is the only AI tool you may use for coursework**, and any AI use must be disclosed. If you used CoPilot anywhere in this notebook, write one sentence at the bottom (`Reflection` section) describing what you used it for. Submitting AI-generated work as your own is a violation of academic integrity.

## Data source and citation

The dataset you will work with comes from a published teaching collection:

> Parsekian, A. (n.d.). *IGUaNA Unit 3: Gravity and Magnetics Field Data Exercises, Part 3a (Stairwell gravity).* Science Education Resource Center, Carleton College. CC-BY-NC-SA 4.0. <https://serc.carleton.edu/iguana/teaching_materials/grav_mag/unit3.html>

These are real gravimeter readings taken in a campus stairwell: open-and-close ground-level reads to track instrument drift, plus one reading at each half-floor as the operator climbed.

## Loading the data

The code below loads `stairwell.csv` automatically from a stable public web address, so in most cases you do not need to download anything: just run the cells.

**If you have no internet, or the link is not live yet,** use the Brightspace fallback:

1. Download `stairwell.csv` from the Brightspace HW1 page.
2. In Google Colab, click the **📁 Files** icon in the left sidebar.
3. Drag `stairwell.csv` into the file panel.
4. In the loading cell below, comment out the `pd.read_csv(DATA_URL)` line and use the commented `pd.read_csv("stairwell.csv")` line instead.

Colab deletes uploaded files when the runtime disconnects; if an uploaded file vanishes, re-run the loading cell (the URL path) or re-upload it.

## Setup

These imports give us NumPy (numbers and arrays), Pandas (tables), and Plotly (interactive plots). All three come pre-installed in Colab; no `pip install` needed.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe

## Part 1: The Anchor Equation (~15 min)

From class: the gravitational acceleration at distance `r` from a point mass `M` is

$$g = \frac{G \, M}{r^2}$$

where:
- `G = 6.674e-11 m³ kg⁻¹ s⁻²` (gravitational constant)
- `M = 5.972e24 kg` (mass of Earth)
- `r` = distance from Earth's center (m)

We will treat the Earth as a point mass, the simplest possible model. Run the cell below to compute `g` at sea level and at the top of a hypothetical 100 m building.

**Question 1.1.** Before you run the cell, *predict*: by how much (in m/s², or in microgal, see below) do you expect `g` to change between sea level and 100 m above sea level? Write your prediction here:

*(Your answer):*

In [ ]:
G = 6.674e-11    # m^3 / (kg s^2)
M_earth = 5.972e24    # kg
R_earth = 6.371e6    # m

g_sea_level = G * M_earth / R_earth**2
g_at_100m = G * M_earth / (R_earth + 100)**2
delta_g = g_sea_level - g_at_100m

print(f'g at sea level:  {g_sea_level:.6f} m/s^2')
print(f'g at 100 m:      {g_at_100m:.6f} m/s^2')
print(f'difference:      {delta_g*1e8:.2f} microgal  (1 microgal = 1e-8 m/s^2)')
print(f'predicted rate:  {delta_g*1e5/100:.4f} mGal per meter   (1 mGal = 1e-5 m/s^2)')

**Question 1.2.** The gravimeter unit milligal (mGal) is the working unit in this assignment. Looking at the printout: how many mGal does `g` change per meter of elevation? Compare to your prediction.

*(Your answer):*

## Part 2: Load the data (~15 min)

The stairwell experiment was simple: the operator stood at ground level, took one reading, then walked up to floor 0.5, took another, kept climbing to floor 4.5, then walked back down to ground level and took one final reading. Those two ground-level readings, at the start and the end, bracket the run and let us see how much the instrument's reading *drifted* over the ~68 minutes of work, even though nothing about Earth's gravity actually changed.

The CSV includes a `minutes_since_start` column: the number of minutes elapsed since the first reading. You will use it in Part 4 as the time axis for the drift fit.

In [ ]:
# Primary path: load directly from a stable public URL (no upload needed).
# Builder: replace the stub with the stairwell.csv URL from the Public URL
# column of assignment_calendar.md once the dataset is hosted.
DATA_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/stairwell.csv/stairwell.csv"
df = pd.read_csv(DATA_URL)

# Fallback (no internet, or URL not live yet): download stairwell.csv from the
# Brightspace HW1 page, drag it into the Colab file panel, then comment out the
# two lines above and use:
# df = pd.read_csv("stairwell.csv")

df.head(15)

In [ ]:
df.describe()

In [ ]:
df.info()

**Question 2.1.** Looking at the `station` column: which two rows are the start-of-run and end-of-run ground-level readings? (Hint: one is labeled `Ground level` and the other has a typo. The typo appears in the source data and is preserved here.)

*(Your answer):*

**Question 2.2.** The `reading_mgal` values are around 3423 mGal. The `g_sea_level` you computed in Part 1 was about 9.8 m/s² ≈ 980,000 mGal. Why is the gravimeter showing a number near 3423 instead of near 980,000? *(Hint: it's a relative instrument, not an absolute one. The 3423 is reading off an arbitrary internal zero-point set when the instrument was calibrated. We only care about *changes* in this number, not the absolute value.)*

*(Your answer):*

## Part 3: Visualize the data (~15 min)

Two plots summarize the dataset.

### 3a. Gravity over time

If we plot the reading against the clock, we expect to see two ground-level readings (start and end) that should be *identical* (same physical place, same Earth) but in practice are slightly different. The difference is the **instrument drift**: spring tension, temperature changes, and tides cause a slow steady creep in the reading.

In [ ]:
fig = px.scatter(
    df,
    x='time',
    y='reading_mgal',
    color='station', symbol='station',
    title='Stairwell gravity readings vs. clock time',
    labels={'time': 'Time (HH:MM:SS)', 'reading_mgal': 'Gravimeter reading (mGal)'},
)
fig.update_traces(marker=dict(size=12))
fig.show()

**Figure description:** A scatter plot with clock time (HH:MM:SS) on the x-axis and gravimeter reading (mGal) on the y-axis; each marker is one station, shown by color. This is the drift view: the start-of-run and end-of-run ground-level reads (the same physical place) sit at the left and right edges, and the small vertical offset between them is the instrument drift that Question 3.1 asks you to read off. Non-visual path: every point's `time`, `reading_mgal`, and `station` are in the data table printed in Part 2, so the two ground-level rows give the drift directly without the plot.

### 3b. Gravity over elevation

Now plot the same readings against the *elevation* of each station above the ground floor. Under the anchor equation `g = GM/r²`, gravity should *decrease* as we move up. The slope of this line, change in g per meter of elevation, is the **free-air gradient**.

In [ ]:
fig = px.scatter(
    df,
    x='elevation_m',
    y='reading_mgal',
    color='station', symbol='station',
    title='Stairwell gravity readings vs. elevation',
    labels={'elevation_m': 'Elevation above ground (m)', 'reading_mgal': 'Gravimeter reading (mGal)'},
)
fig.update_traces(marker=dict(size=12))
fig.show()

**Figure description:** A scatter plot with elevation above ground (meters) on the x-axis and gravimeter reading (mGal) on the y-axis; each marker is one station, shown by color. This is the free-air view: it shows how the reading changes as the operator climbs the stairwell, which Question 3.2 asks you to describe and compare with g = GM/r². Non-visual path: the `elevation_m` and `reading_mgal` columns in the Part 2 data table give every plotted value without the plot.

**Question 3.1.** Look at plot 3a. The two ground-level readings (`Ground level` at the start, `gound level` at the end) sit at almost the same `y` but not exactly. By how many mGal did the instrument drift over the run? Is the drift positive or negative?

*(Your answer):*

**Question 3.2.** Look at plot 3b. As elevation increases, what happens to the reading? Is this consistent with `g = GM/r²`?

*(Your answer):*

## Part 4: Correct the drift, then measure the free-air gradient (~45 min)

Plot 3b looks like a clean line, so it is tempting to fit a slope straight through the climbing readings and call that the free-air gradient. **The raw reads still contain drift; correct the drift first.** The instrument drifted the whole time the operator was climbing, so every climbing read carries some drift on top of the real elevation signal. Throwing out the two ground reads does not remove that drift; it only removes the two points that let us *measure* it.

The correct procedure has three steps:

1. **Fit the drift.** The two ground-level reads are at the same physical place (elevation 0), so any difference between them is pure drift. Fit a straight line to `reading_mgal` vs. `minutes_since_start` through *just those two ground reads*. The slope is the instrument's drift rate in mGal per minute.
2. **Subtract the drift from every reading.** Using that drift line, remove `slope × minutes_since_start` from all eleven readings. After this, the two ground reads collapse onto the same value, and every climbing read has had its time-dependent drift removed.
3. **Fit the corrected gravity vs. elevation.** Now the slope of `drift_corrected` vs. `elevation_m` is the free-air gradient, with the drift no longer leaking into it.

We use `numpy.polyfit(x, y, 1)` for the line fits. It returns `[slope, intercept]`.

### Step 1: Fit the drift through the two ground reads

In [ ]:
# Step 1: the two ground-level reads (same place, elevation 0) isolate the drift.
ground = df[df['station'].isin(['Ground level', 'gound level'])]

drift_slope, drift_intercept = np.polyfit(
    ground['minutes_since_start'], ground['reading_mgal'], 1
)

print(f'drift slope:     {drift_slope:+.6f} mGal per minute')
print(f'over the {df["minutes_since_start"].max():.0f}-minute run, total drift: '
      f'{drift_slope * df["minutes_since_start"].max():+.4f} mGal')

### Step 2: Subtract the drift from every reading and replot

Apply the drift line to all eleven readings: `drift_corrected = reading_mgal - drift_slope × minutes_since_start`. The plot below shows the raw readings and the drift-corrected readings on the same elevation axis so you can see what the correction did to the two ground reads.

In [ ]:
# Step 2: subtract the drift line from EVERY reading (not just the climbing ones).
df['drift_corrected'] = df['reading_mgal'] - drift_slope * df['minutes_since_start']

# Confirm the two ground reads now agree (drift removed):
print('ground reads after drift correction (should be nearly equal):')
print(df[df['station'].isin(['Ground level', 'gound level'])]
      [['station', 'reading_mgal', 'drift_corrected']].to_string(index=False))

# Replot raw vs. drift-corrected gravity against elevation.
# Color AND marker symbol both distinguish the two series (readable in grayscale).
plot_df = df.melt(
    id_vars=['station', 'elevation_m'],
    value_vars=['reading_mgal', 'drift_corrected'],
    var_name='series',
    value_name='gravity_mgal',
)
fig = px.scatter(
    plot_df,
    x='elevation_m',
    y='gravity_mgal',
    color='series',
    symbol='series',
    color_discrete_sequence=px.colors.qualitative.Safe,
    title='Raw vs. drift-corrected gravity vs. elevation',
    labels={
        'elevation_m': 'Elevation above ground (m)',
        'gravity_mgal': 'Gravimeter reading (mGal)',
        'series': 'Series',
    },
)
fig.update_traces(marker=dict(size=11))
fig.show()

**Figure description:** A scatter plot with elevation above ground (meters, 0 to ~16) on the x-axis and gravimeter reading (mGal) on the y-axis. Two traces are shown and distinguished by both color and marker symbol: the raw `reading_mgal` (circles) and the `drift_corrected` reading (diamonds). The two ground-level points (at elevation 0) are separated in the raw trace but sit on top of each other in the drift-corrected trace. To read off the free-air gradient without the plot, run the next cell, which prints the fitted slope and its uncertainty.

### Step 3: Fit the drift-corrected gravity vs. elevation

Now that the drift is gone, the slope of `drift_corrected` vs. `elevation_m` is the free-air gradient. We fit it through the climbing reads (the two ground reads are both at elevation 0 and only served to anchor the drift). We also report a **1-sigma uncertainty** on the slope using `numpy.polyfit(..., cov=True)`, so we can say how well the data pin the gradient down, not just a single number.

In [ ]:
# Step 3: fit the free-air gradient on the DRIFT-CORRECTED gravity.
# Fit through the climbing reads (the ground reads are both at elevation 0 and only
# anchor the drift); report the slope with a 1-sigma uncertainty from the fit.
climb = df[~df['station'].isin(['Ground level', 'gound level'])]

# np.polyfit with cov=True returns the covariance matrix; sqrt of its diagonal is 1 sigma.
coeffs, cov = np.polyfit(
    climb['elevation_m'], climb['drift_corrected'], 1, cov=True
)
gradient, intercept = coeffs
gradient_sigma = np.sqrt(cov[0, 0])

print(f'free-air gradient: {gradient:.4f} +/- {gradient_sigma:.4f} mGal per meter (1 sigma)')
print(f'canonical free-air gradient:                 {-0.3086:.4f} mGal per meter')
print()
# How many sigma is the measured gradient from the canonical value?
n_sigma = abs(gradient - (-0.3086)) / gradient_sigma
print(f'difference from canonical: {n_sigma:.1f} sigma '
      f'(a real, resolved gap -- see the self-check below)')

**Self-check.** A correct drift-then-fit on this dataset gives a free-air gradient of about **-0.287 ± 0.008 mGal/m** (your numbers should round to roughly this, sign and all). That is close to but not exactly the canonical free-air gradient of **-0.3086 mGal/m**, and with a 1-sigma uncertainty of ~0.008, the gap of ~0.022 mGal/m is roughly 2.7 sigma (about 3 sigma): a statistically resolved difference, not measurement noise. The gap is real, not a bug; Q4.2 and Q5.2 ask you to account for it. If your fitted gradient is positive, or far from -0.287, recheck that you subtracted the drift (Step 2) before fitting (Step 3), and that you fit `drift_corrected` against `elevation_m`.

**Question 4.1.** Report your drift slope (Step 1) in mGal per minute, and its sign. In one sentence, what does that sign mean physically about how the instrument's reading changed over the hour?

*(Your answer):*

**Question 4.2.** Report your free-air gradient and its 1-sigma uncertainty (Step 3), e.g. "-0.287 ± 0.008 mGal/m". The fitted gradient sits about 2.7 sigma (about 3 sigma) from the canonical -0.3086 mGal/m, so the gap is statistically resolved, not measurement noise. Give one physical reason the gradient measured *inside a stairwell* would be smaller in magnitude than the open-air value. (Hint: think about what mass is between the gravimeter and open air.)

*(Your answer):*

**Question 4.3.** Why is it wrong to fit the free-air gradient on the *raw* climbing reads, even after dropping the two ground reads? What specifically does dropping the ground reads remove, and what does it fail to remove?

*(Your answer):*

**Question 4.4.** *Forecast.* If we returned to the same building tomorrow at the same time of day and read the gravimeter at ground level, would you expect to get the same number, the start-of-run number, the end-of-run number, or something different? Defend your answer in one or two sentences. (This question has no single right answer; show your reasoning.)

*(Your answer):*

## Part 5: Math: derive the free-air gradient (~20 min)

In Part 4 you *measured* a free-air gradient near -0.3 mGal/m. Where does that number come from? It falls straight out of the anchor equation using the binomial expansion on the Taylor & Binomial card (in the Math Reference Cards PDF on the Resources page).

The binomial expansion to first order is

$$(1 + x)^n \approx 1 + n\,x \qquad \text{for small } x.$$

Start from gravity at height $h$ above Earth's surface (radius $R$, mass $M$):

$$g(R+h) = \frac{GM}{(R+h)^2} = \frac{GM}{R^2}\left(1 + \frac{h}{R}\right)^{-2}.$$

Here the small parameter is $x = h/R$ (a 100 m building over a 6371 km Earth is $x \approx 1.6\times10^{-5}$, very small), and the exponent is $n = -2$. Applying the expansion:

$$g(R+h) \approx \frac{GM}{R^2}\left(1 - 2\,\frac{h}{R}\right) = g_0\left(1 - \frac{2h}{R}\right),$$

where $g_0 = GM/R^2$ is the surface value. So gravity falls off *linearly* with height near the surface, and the slope (the free-air gradient) is

$$\frac{dg}{dh} \approx -\frac{2\,g_0}{R}.$$

Run the cell below to plug in the numbers and compare to the canonical -0.3086 mGal/m.

In [ ]:
G = 6.674e-11    # m^3 / (kg s^2)
M_earth = 5.972e24    # kg
R_earth = 6.371e6    # m

g0 = G * M_earth / R_earth**2
free_air_gradient = -2 * g0 / R_earth        # m/s^2 per meter of elevation
free_air_gradient_mgal = free_air_gradient * 1e5    # 1 mGal = 1e-5 m/s^2

print(f'g0 (surface gravity):         {g0:.4f} m/s^2')
print(f'free-air gradient:            {free_air_gradient:.4e} m/s^2 per meter')
print(f'free-air gradient (mGal/m):   {free_air_gradient_mgal:.4f}')
print()
print(f'canonical value:              {-0.3086:.4f} mGal/m')

**Question 5.1.** The derivation gave you a free-air gradient near -0.308 mGal/m purely from $g = GM/r^2$, with no field data at all. What was the small parameter in the binomial expansion, and roughly how small is it for a 100 m building? Why does that justify keeping only the first-order term?

*(Your answer):*

**Question 5.2.** Compare the derived gradient (this part) to the gradient you *measured* from the stairwell data (Part 4, about -0.287 mGal/m). They are close but not equal. Which one is the "true" free-air gradient, and what does the difference between them tell you about the stairwell environment?

*(Your answer):*

**Question 5.3.** The expansion is to *first order*. If you kept the second-order term, would it bend the gravity-vs-height curve so the magnitude of the gradient grows or shrinks with height? You do not need to compute it; just sign the answer and justify in one sentence.

*(Your answer):*

## Part 6: Reflection (~10 min)

The anchor equation is

$$g = \frac{G \, M}{r^2}$$

It assumes Earth is a point mass, the observer is at the reference surface, there's nothing between the observer and the reference, and the instrument is stable in time. Each of those assumptions becomes a *correction* later in this course.

**Question 6.1.** Which two of those four assumptions did the stairwell dataset just *break*? Cite the specific feature of the data that broke each assumption.

*(Your answer):*

**Question 6.2.** In one or two sentences, explain why the stairwell experiment used **two** ground-level readings (one at the start, one at the end) instead of just one.

*(Your answer):*

**Question 6.3.** *(Optional, ungraded)* What was the most surprising thing about this dataset for you?

*(Your answer):*

## Honor pledge

If you used CUNY CoPilot anywhere in this notebook, write one sentence describing what you used it for:

*(Your disclosure, or write "I did not use AI for this assignment."):*

## How to submit

1. Run all cells from the top. (`Runtime → Run all` in Colab.)
2. Make sure all your short answers are filled in.
3. `File → Download → Download .ipynb`.
4. Rename the file to `HW1_LASTNAME.ipynb` (e.g., `HW1_Smith.ipynb`).
5. Upload to the Brightspace HW1 dropbox by **Wednesday September 23, 11:59 PM**.

If something is broken or unclear, post on the **Ask the Class (General Q&A)** discussion topic. Other students may have the same question.